[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/practice/advanced/02_deep_learning_internals.ipynb)

# Luyện tập nâng cao 2 — Bên trong deep learning (NumPy)

Cài lại các khối cơ bản của mạng nơ-ron bằng NumPy và kiểm tra bằng công cụ độc lập (SciPy, sai phân số). Không cần GPU hay PyTorch — mục tiêu là hiểu **chính xác** mỗi khối tính gì.

Chương liên quan: 27, 28, 29, 30, 31.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/practice/advanced

In [ ]:
import numpy as np
from scipy.signal import correlate2d

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

rng = np.random.default_rng(0)

**Bài 1 — Tích chập 2D.** `conv2d(image, kernel, stride=1, padding=0)` (một kênh, "valid" khi padding = 0, phép tương quan chéo như trong deep learning). Kiểm tra với `scipy.signal.correlate2d`.

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    ### START CODE HERE ### (≈ 8 dòng)
    # STARTER: return np.zeros((1, 1))
    img = np.pad(image, padding)
    kh, kw = kernel.shape
    oh = (img.shape[0] - kh) // stride + 1
    ow = (img.shape[1] - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(img[i * stride:i * stride + kh, j * stride:j * stride + kw] * kernel)
    return out
    ### END CODE HERE ###


image, kernel = rng.normal(size=(9, 9)), rng.normal(size=(3, 3))

In [ ]:
check_close(conv2d(image, kernel), correlate2d(image, kernel, mode="valid"), message="khớp scipy (valid)")
check_equal(conv2d(image, kernel, padding=1).shape, (9, 9), "padding 1 giữ kích thước")
check_equal(conv2d(image, kernel, stride=2).shape, (4, 4), "(9 − 3)/2 + 1 = 4")
check_close(conv2d(image, kernel, padding=1), correlate2d(image, kernel, mode="same"), message="khớp scipy (same)")

**Bài 2 — Max pooling 2×2 và gradient của nó.** `maxpool_forward(x)` (x: H×W chẵn) → (đầu ra, mặt nạ vị trí max); `maxpool_backward(grad_out, mask)` → gradient theo x: gradient chỉ chảy về **vị trí max** của mỗi ô.

In [ ]:
def maxpool_forward(x):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return x, np.ones_like(x, bool)
    H, W = x.shape
    blocks = x.reshape(H // 2, 2, W // 2, 2).transpose(0, 2, 1, 3).reshape(H // 2, W // 2, 4)
    out = blocks.max(axis=-1)
    mask = (blocks == out[..., None]).reshape(H // 2, W // 2, 2, 2).transpose(0, 2, 1, 3).reshape(H, W)
    return out, mask
    ### END CODE HERE ###


def maxpool_backward(grad_out, mask):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return np.zeros(mask.shape)
    return np.repeat(np.repeat(grad_out, 2, axis=0), 2, axis=1) * mask
    ### END CODE HERE ###


x2 = np.array([[1.0, 2, 5, 0], [3, 0, 1, 2], [0, 7, 2, 2.5], [1, 1, 3, 1]])
out2, mask2 = maxpool_forward(x2)
print(out2)

In [ ]:
check_close(out2, [[3, 5], [7, 3]], message="max của mỗi ô 2×2")
grad = maxpool_backward(np.array([[1.0, 2], [3, 4]]), mask2)
check_close(grad, [[0, 0, 2, 0], [1, 0, 0, 0], [0, 3, 0, 0], [0, 0, 4, 0]], message="gradient chỉ về vị trí max")

**Bài 3 — Backprop và kiểm tra gradient.** Mạng `x → Linear(4, 8) → tanh → Linear(8, 1)`, loss MSE trung bình. Viết `forward_backward(params, X, y)` → `(loss, grads)` (dict cùng khoá `W1, b1, W2, b2`). Kiểm tra từng gradient bằng sai phân trung tâm.

In [ ]:
params = {"W1": rng.normal(size=(4, 8)) * 0.5, "b1": rng.normal(size=8) * 0.1, "W2": rng.normal(size=(8, 1)) * 0.5, "b2": np.zeros(1)}
X3, y3 = rng.normal(size=(16, 4)), rng.normal(size=16)


def forward_backward(params, X, y):
    ### START CODE HERE ### (≈ 10 dòng)
    # STARTER: return 0.0, {k: np.zeros_like(v) for k, v in params.items()}
    Z1 = X @ params["W1"] + params["b1"]
    A1 = np.tanh(Z1)
    pred = (A1 @ params["W2"] + params["b2"])[:, 0]
    loss = float(np.mean((pred - y) ** 2))
    d_pred = 2 * (pred - y)[:, None] / len(y)
    grads = {"W2": A1.T @ d_pred, "b2": d_pred.sum(axis=0)}
    dZ1 = (d_pred @ params["W2"].T) * (1 - A1 ** 2)
    grads.update({"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0)})
    return loss, grads
    ### END CODE HERE ###


def numeric_grad(name, h=1e-6):
    g = np.zeros_like(params[name])
    for idx in np.ndindex(g.shape):
        old = params[name][idx]
        params[name][idx] = old + h
        up = forward_backward(params, X3, y3)[0]
        params[name][idx] = old - h
        down = forward_backward(params, X3, y3)[0]
        params[name][idx] = old
        g[idx] = (up - down) / (2 * h)
    return g


_, analytic = forward_backward(params, X3, y3)
rel_errors = {k: np.abs(analytic[k] - numeric_grad(k)).max() / (np.abs(analytic[k]).max() + 1e-12) for k in params}
print({k: f"{v:.1e}" for k, v in rel_errors.items()})

In [ ]:
check_true(all(v < 1e-6 for v in rel_errors.values()), "gradient giải tích khớp gradient số (sai số tương đối < 1e-6)")

**Bài 4 — Layer normalization.** `layer_norm(x, gamma, beta, eps=1e-5)`: chuẩn hoá **theo chiều đặc trưng của từng mẫu** (trục cuối), rồi `gamma * x̂ + beta`. Khác BatchNorm (chuẩn hoá theo batch) thế nào?

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return x
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mean) / np.sqrt(var + eps) + beta
    ### END CODE HERE ###


x4 = rng.normal(5, 3, size=(6, 10))
y4 = layer_norm(x4, np.ones(10), np.zeros(10))

In [ ]:
check_close(y4.mean(axis=1), np.zeros(6), tol=1e-9, message="mỗi mẫu có trung bình 0")
check_close(y4.std(axis=1), np.ones(6), tol=1e-4, message="mỗi mẫu có độ lệch chuẩn 1")
check_close(layer_norm(x4[:1], np.ones(10), np.zeros(10)), y4[:1], message="kết quả một mẫu không phụ thuộc các mẫu khác trong batch")

LayerNorm chuẩn hoá trong **từng mẫu** nên không phụ thuộc kích thước batch, hoạt động giống nhau lúc train và lúc suy luận — phù hợp với chuỗi độ dài thay đổi (Transformer). BatchNorm chuẩn hoá theo **batch** cho từng kênh, cần thống kê chạy (running stats) cho lúc suy luận và kém ổn định với batch nhỏ.

**Bài 5 — Multi-head self-attention.** `multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=False)`: X (T × d), chiếu Q, K, V (d × d), tách thành `n_heads` đầu (mỗi đầu d/n_heads chiều), attention có tỉ lệ $1/\sqrt{d_h}$ (mặt nạ nhân quả nếu `causal`), ghép các đầu, nhân `Wo`. Trả về `(output (T × d), weights (n_heads × T × T))`.

In [ ]:
def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=False):
    ### START CODE HERE ### (≈ 12 dòng)
    # STARTER: return X, np.zeros((n_heads, len(X), len(X)))
    T, d = X.shape
    dh = d // n_heads
    split = lambda M: M.reshape(T, n_heads, dh).transpose(1, 0, 2)  # noqa: E731
    Q, K, V = split(X @ Wq), split(X @ Wk), split(X @ Wv)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(dh)
    if causal:
        scores = np.where(np.tril(np.ones((T, T), bool)), scores, -np.inf)
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)
    heads = weights @ V
    return heads.transpose(1, 0, 2).reshape(T, d) @ Wo, weights
    ### END CODE HERE ###


T, d, h = 5, 8, 2
X5 = rng.normal(size=(T, d))
Wq, Wk, Wv, Wo = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(4))
out5, w5 = multi_head_attention(X5, Wq, Wk, Wv, Wo, h, causal=True)

In [ ]:
check_equal(out5.shape, (5, 8), "đầu ra T × d")
check_equal(w5.shape, (2, 5, 5), "trọng số mỗi đầu T × T")
check_close(w5.sum(axis=-1), np.ones((2, 5)), message="mỗi hàng trọng số cộng bằng 1")
check_close(np.triu(w5, 1), np.zeros((2, 5, 5)), message="mặt nạ nhân quả")
X5_changed = X5.copy()
X5_changed[-1] += 10
check_close(multi_head_attention(X5_changed, Wq, Wk, Wv, Wo, h, causal=True)[0][:-1], out5[:-1], message="đổi token cuối không ảnh hưởng các vị trí trước (nhân quả)")

**Bài 6 — Positional encoding hình sin.** `positional_encoding(T, d)`: $PE_{t, 2i} = \sin(t/10000^{2i/d})$, $PE_{t, 2i+1} = \cos(t/10000^{2i/d})$. Kiểm tra hai tính chất: mỗi vị trí có chuẩn $\sqrt{d/2}$; và tích vô hướng $PE_t \cdot PE_{t+k}$ chỉ phụ thuộc **khoảng cách** $k$, không phụ thuộc $t$.

In [ ]:
def positional_encoding(T, d):
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return np.zeros((T, d))
    pos = np.arange(T)[:, None]
    freq = 1 / 10000 ** (np.arange(0, d, 2) / d)
    pe = np.zeros((T, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(pos * freq), np.cos(pos * freq)
    return pe
    ### END CODE HERE ###


pe = positional_encoding(50, 16)

In [ ]:
check_close(np.linalg.norm(pe, axis=1), np.full(50, np.sqrt(8)), message="chuẩn mỗi vị trí = √(d/2)")
check_close(pe[3] @ pe[8], pe[20] @ pe[25], message="tích vô hướng chỉ phụ thuộc khoảng cách (5)")
check_greater(pe[10] @ pe[11], pe[10] @ pe[30], "vị trí gần nhau giống nhau hơn")